# Лабораторная работа 2. Классификация. Нейронные сети

## Введение

### Цели и задачи

Цель работы: изучение методов классификации данных, реализованные в библиотеке Scikit-Learn, а также ознакомление с нейронными сетями с использованием библиотек TensorFlow и TensorBoard. 

Постановка задачи:

1. Необходимо выбрать и подготовить датасет для классификации, затем построить классификационные модели с помощью пяти методов:

    a. Наивный Байесовский классификатор (GaussianNB, MultinomialNB, ComplementNB, BernoulliNB);

    b. Деревья решений (Decision Tree);

    c. Линейный дискриминантный анализ (Linear Discriminant Analysis);

    d. Метод опорных векторов (Support Vector Machines);

    e. Метод ближайших соседей (k-Nearest Neighbors);

2. Затем сравните качество работы классификаторов с помощью следующих метрик:

    a. Accuracy (точность),

    b. Precision (доля правильно классифицированных положительных примеров),

    c. Recall (чувствительность, доля верно найденных положительных примеров),

    d. F1-Score (среднее гармоническое precision и recall),

    e. Площадь под кривой ROC (AUC-ROC).

3. Настройте гиперпараметры каждого метода и проведите исследование влияния различных параметров на качество классификации.

4. Реализуйте и протестируйте нейронную сеть на TensorFlow, исследуйте эффект настройки гиперпараметров и визуализируйте процесс обучения с помощью инструмента TensorBoard.

## Выбор и подготовка датасета

### Информация о выбранном датасете

В работе используется датасет Indian Liver Patient Dataset. Это реальные клинические данные 583 пациентов из северо-восточной части штата Андхра-Прадеш (Индия) - результаты лабораторных анализов крови. По результатам анализов крови определить, есть ли у пациента заболевание печени. 



1. `Age` - возраст пациента, лет  
2. `Gender` - пол пациента (Male/Female)   
3. `Total_Bilirubin` - общий билирубин, мг/дл   
4. `Direct_Bilirubin` - прямой (связанный) билирубин, мг/дл   
5. `Alkaline_Phosphotase` - щелочная фосфатаза, ед/л  
6. `Alamine_Aminotransferase` - аланинаминотрансфераза (АЛТ), ед/л  
7. `Aspartate_Aminotransferase` - аспартатаминотрансфераза (АСТ), ед/л   
8. `Total_Protiens` - общий белок, г/дл   
9. `Albumin` - альбумин, г/дл   
10. `Albumin_and_Globulin_Ratio` - отношение альбумина к глобулину   
11. `Dataset` - **целевая переменная**: `1` - заболевание печени, `2` - нет заболевания    
Итого: 10 признаков (9 числовых + пол) и 1 целевая переменная, 583 объекта.

### Импорт библиотек

In [19]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB, MultinomialNB, ComplementNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

### Загрузка и первичный осмотр данных

In [20]:
df = pd.read_csv('indian_liver_patient.csv')
print('Размер таблицы (строк, столбцов):', df.shape)
df.head(10)

Размер таблицы (строк, столбцов): (583, 11)


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,1
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490,60,68,7.0,3.3,0.89,1
3,58,Male,1.0,0.4,182,14,20,6.8,3.4,1.00,1
4,72,Male,3.9,2.0,195,27,59,7.3,2.4,0.40,1
5,46,Male,1.8,0.7,208,19,14,7.6,4.4,1.30,1
6,26,Female,0.9,0.2,154,16,12,7.0,3.5,1.00,1
7,29,Female,0.9,0.3,202,14,11,6.7,3.6,1.10,1
8,17,Male,0.9,0.3,202,22,19,7.4,4.1,1.20,2
9,55,Male,0.7,0.2,290,53,58,6.8,3.4,1.00,1


In [18]:
df.info()

df.describe().T.round(2)

<class 'pandas.DataFrame'>
RangeIndex: 583 entries, 0 to 582
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         583 non-null    int64  
 1   Gender                      583 non-null    str    
 2   Total_Bilirubin             583 non-null    float64
 3   Direct_Bilirubin            583 non-null    float64
 4   Alkaline_Phosphotase        583 non-null    int64  
 5   Alamine_Aminotransferase    583 non-null    int64  
 6   Aspartate_Aminotransferase  583 non-null    int64  
 7   Total_Protiens              583 non-null    float64
 8   Albumin                     583 non-null    float64
 9   Albumin_and_Globulin_Ratio  579 non-null    float64
 10  Dataset                     583 non-null    int64  
dtypes: float64(5), int64(5), str(1)
memory usage: 50.2 KB


,count,mean,std,min,25%,50%,75%,max
Age,583.0,44.75,16.19,4.0,33.0,45.00,58.0,90.0
Total_Bilirubin,583.0,3.30,6.21,0.4,0.8,1.00,2.6,75.0
Direct_Bilirubin,583.0,1.49,2.81,0.1,0.2,0.30,1.3,19.7
Alkaline_Phosphotase,583.0,290.58,242.94,63.0,175.5,208.00,298.0,2110.0
Alamine_Aminotransferase,583.0,80.71,182.62,10.0,23.0,35.00,60.5,2000.0
Aspartate_Aminotransferase,583.0,109.91,288.92,10.0,25.0,42.00,87.0,4929.0
Total_Protiens,583.0,6.48,1.09,2.7,5.8,6.60,7.2,9.6
Albumin,583.0,3.14,0.80,0.9,2.6,3.10,3.8,5.5
Albumin_and_Globulin_Ratio,579.0,0.95,0.32,0.3,0.7,0.93,1.1,2.8
Dataset,583.0,1.29,0.45,1.0,1.0,1.00,2.0,2.0


### Первичный анализ качества данных